# Credit Card Fraud Detection
Detecting fraudulent transactions using machine learning on a highly imbalanced dataset.

**Dataset:** [Kaggle — Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Models:** Random Forest, XGBoost, Logistic Regression, Decision Tree  
**Imbalance Handling:** SMOTE oversampling, class_weight='balanced', scale_pos_weight

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print('Libraries loaded successfully.')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('creditcard.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Class Imbalance Analysis

In [ ]:
class_counts = df['Class'].value_counts()
class_pct    = df['Class'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(f'  Legitimate (0): {class_counts[0]:,}  ({class_pct[0]:.4f}%)')
print(f'  Fraud      (1): {class_counts[1]:,}  ({class_pct[1]:.4f}%)')
print(f'  Imbalance Ratio: {class_counts[0] // class_counts[1]}:1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], class_counts.values,
            color=['steelblue', 'salmon'], edgecolor='white')
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie(class_counts.values, labels=['Legitimate', 'Fraud'],
            autopct='%1.3f%%', colors=['steelblue', 'salmon'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Class Distribution (%)', fontweight='bold')

plt.suptitle('Highly Imbalanced Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Data Cleaning

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())

df.drop_duplicates(inplace=True)
print('Shape after removing duplicates:', df.shape)

## 5. Exploratory Data Analysis

In [ ]:
# Amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df[df['Class'] == 0]['Amount'].hist(bins=60, ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('Transaction Amount — Legitimate', fontweight='bold')
axes[0].set_xlabel('Amount')

df[df['Class'] == 1]['Amount'].hist(bins=60, ax=axes[1], color='salmon', alpha=0.8)
axes[1].set_title('Transaction Amount — Fraud', fontweight='bold')
axes[1].set_xlabel('Amount')

plt.tight_layout()
plt.show()

In [ ]:
# Top features correlated with fraud
corr = df.corr()['Class'].drop('Class').sort_values()

plt.figure(figsize=(10, 5))
colors = ['salmon' if v < 0 else 'steelblue' for v in corr.values]
corr.plot(kind='bar', color=colors)
plt.title('Feature Correlation with Fraud (Class)', fontweight='bold')
plt.ylabel('Correlation Coefficient')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 6. Feature Scaling & Data Split

In [ ]:
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])
df['Time']   = scaler.fit_transform(df[['Time']])

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y
)

print('Train size:', X_train.shape, '| Test size:', X_test.shape)
print('Train fraud count:', y_train.sum(), '| Test fraud count:', y_test.sum())

## 7. Handle Class Imbalance with SMOTE

In [ ]:
smote = SMOTE(random_state=1)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('Before SMOTE:', dict(zip(*np.unique(y_train, return_counts=True))))
print('After  SMOTE:', dict(zip(*np.unique(y_train_sm, return_counts=True))))

## 8. Model Training

In [ ]:
# --- Random Forest (without SMOTE) ---
rf_model = RandomForestClassifier(n_estimators=10, max_depth=10, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print('Random Forest (without SMOTE)')
print(classification_report(y_test, rf_pred))

In [ ]:
# --- Random Forest (with SMOTE) ---
rf_smote = RandomForestClassifier(n_estimators=10, max_depth=10, n_jobs=-1, random_state=42)
rf_smote.fit(X_train_sm, y_train_sm)
rf_smote_pred = rf_smote.predict(X_test)

print('Random Forest (with SMOTE)')
print(classification_report(y_test, rf_smote_pred))

In [ ]:
# --- XGBoost (scale_pos_weight handles imbalance) ---
ratio = int((y_train == 0).sum() / (y_train == 1).sum())
print(f'scale_pos_weight used: {ratio}')

xgb_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=ratio, random_state=42, eval_metric='aucpr'
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

print('XGBoost')
print(classification_report(y_test, xgb_pred))

In [ ]:
# --- Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print('Logistic Regression')
print(classification_report(y_test, lr_pred))

In [ ]:
# --- Decision Tree ---
dt_model = DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

print('Decision Tree')
print(classification_report(y_test, dt_pred))

## 9. Model Comparison

In [ ]:
def get_metrics(name, y_true, y_pred, y_proba):
    return {
        'Model':      name,
        'Accuracy':   round(accuracy_score(y_true, y_pred), 4),
        'Precision':  round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':     round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1 Score':   round(f1_score(y_true, y_pred, zero_division=0), 4),
        'ROC-AUC':    round(roc_auc_score(y_true, y_proba), 4),
        'Avg Precision': round(average_precision_score(y_true, y_proba), 4),
    }

results = pd.DataFrame([
    get_metrics('RF (no SMOTE)',  y_test, rf_pred,       rf_model.predict_proba(X_test)[:, 1]),
    get_metrics('RF (SMOTE)',     y_test, rf_smote_pred, rf_smote.predict_proba(X_test)[:, 1]),
    get_metrics('XGBoost',        y_test, xgb_pred,      xgb_model.predict_proba(X_test)[:, 1]),
    get_metrics('Logistic Reg.',  y_test, lr_pred,        lr_model.predict_proba(X_test)[:, 1]),
    get_metrics('Decision Tree',  y_test, dt_pred,        dt_model.predict_proba(X_test)[:, 1]),
])

results.set_index('Model', inplace=True)
results.style.highlight_max(color='lightgreen', axis=0)

In [ ]:
# Bar chart comparison
ax = results[['Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'Avg Precision']].plot(
    kind='bar', figsize=(13, 5), colormap='Set2', edgecolor='white', width=0.75
)
ax.set_title('Model Comparison — Key Metrics', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_xticklabels(results.index, rotation=15, ha='right')
ax.legend(loc='lower right', fontsize=9)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=7, padding=2)
plt.tight_layout()
plt.show()

## 10. ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))

models_proba = [
    ('RF (no SMOTE)',  rf_model.predict_proba(X_test)[:, 1]),
    ('RF (SMOTE)',     rf_smote.predict_proba(X_test)[:, 1]),
    ('XGBoost',        xgb_model.predict_proba(X_test)[:, 1]),
    ('Logistic Reg.',  lr_model.predict_proba(X_test)[:, 1]),
    ('Decision Tree',  dt_model.predict_proba(X_test)[:, 1]),
]

for name, proba in models_proba:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name}  (AUC={auc:.3f})')

plt.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

## 11. Precision-Recall Curves
> For imbalanced datasets, Precision-Recall curves are more informative than ROC curves.

In [ ]:
plt.figure(figsize=(8, 6))

for name, proba in models_proba:
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    plt.plot(recall, precision, linewidth=2, label=f'{name}  (AP={ap:.3f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve Comparison', fontsize=13, fontweight='bold')
plt.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

## 12. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

configs = [
    ('RF (no SMOTE)',  rf_pred,       'Blues'),
    ('RF (SMOTE)',     rf_smote_pred, 'Purples'),
    ('XGBoost',        xgb_pred,      'Oranges'),
    ('Logistic Reg.',  lr_pred,       'Greens'),
    ('Decision Tree',  dt_pred,       'Reds'),
]

for i, (name, preds, cmap) in enumerate(configs):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=axes[i],
                xticklabels=['Legit', 'Fraud'],
                yticklabels=['Legit', 'Fraud'], cbar=False)
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

axes[-1].axis('off')
plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Feature Importance (XGBoost)

In [ ]:
feat_imp = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 5))
feat_imp.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 15 Feature Importances — XGBoost', fontweight='bold')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 14. Cross-Validation (Best Model)

In [ ]:
# Stratified K-Fold ensures fraud cases appear in every fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=ratio, random_state=42, eval_metric='aucpr'
)

cv_scores = cross_val_score(best_model, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)

print('XGBoost — 5-Fold Stratified Cross-Validation (ROC-AUC)')
print(f'  Scores : {np.round(cv_scores, 4)}')
print(f'  Mean   : {cv_scores.mean():.4f}')
print(f'  Std Dev: {cv_scores.std():.4f}')

## 15. Live Prediction Test

In [ ]:
# Sample transaction from dataset (first row — legitimate)
input_data = np.array([[-1.3598071336738, -0.0727811733098497, 2.53634673796914,
                          1.37815522427443, -0.338320769942518, 0.462387777762292,
                          0.239598554061257, 0.0986979012610507, 0.363786969611213,
                          0.0907941719789316, -0.551599533260813, -0.617800855762348,
                         -0.991389847235408, -0.311169353166658, 1.46817697209427,
                         -0.470400525259478, 0.207971241929242, 0.0257905801985591,
                          0.403992960255733, 0.251412098239705, -0.018306777944153,
                          0.277837575558899, -0.110473910188767, 0.0669280749146731,
                          0.128539358273528, -0.189114843888824, 0.133558376740387,
                         -0.0210530534538215, 0.0, 0.0]])  # Time + Amount (scaled)

prediction   = xgb_model.predict(input_data)
probability  = xgb_model.predict_proba(input_data)

print('=== Prediction Result ===')
print('Legitimate Transaction' if prediction[0] == 0 else 'FRAUD DETECTED!')
print(f'Fraud Probability      : {probability[0][1]*100:.4f}%')
print(f'Legitimate Probability : {probability[0][0]*100:.4f}%')

## 16. Summary

| Model | Key Imbalance Strategy | Notes |
|---|---|---|
| Random Forest (no SMOTE) | None | Baseline |
| Random Forest (SMOTE) | Oversampling minority class | Better recall on fraud |
| XGBoost | `scale_pos_weight` | Best overall AUC |
| Logistic Regression | `class_weight='balanced'` | Fast, interpretable |
| Decision Tree | `class_weight='balanced'` | Simple but prone to overfitting |

**Key Takeaway:** For fraud detection, **Recall on the fraud class** is more critical than overall accuracy — missing a fraud (False Negative) is far costlier than a false alarm. XGBoost with `scale_pos_weight` and Precision-Recall AUC evaluation is the recommended approach.